In [ ]:
!pip -q install sentence-transformers faiss-cpu pypdf gradio datasets ollama
!apt-get -qq update && apt-get -qq install -y pciutils lshw zstd
!curl -fsSL https://ollama.com/install.sh | sh

In [2]:
import subprocess, time

subprocess.Popen(["ollama", "serve"],
                 stdout=open("/kaggle/working/ollama.log", "w"),
                 stderr=subprocess.STDOUT)
time.sleep(8)

LLM = "qwen2.5:7b"          # nếu thiếu VRAM/chậm: "qwen2.5:3b"
!ollama pull {LLM}

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest 
pulling 2bada8a74506:   1% ▕                  ▏  23 MB/4.7 GB                  pulling manifest 
pulling 2bada8a74506:   2% ▕                  ▏ 105 MB/4.7 GB                  pulling manifest 
pulling 2bada8a74506:   3% ▕                  ▏ 148 MB/4.7 GB                  pulling manifest 
pulling 2bada8a74506:   5% ▕                  ▏ 254 MB/4.7 GB                  pulling manifest 
pulling 2bada8a74506:   8% ▕█                 ▏ 360 MB/4.7 GB                  pulling manifest 
pulling 2bada8a74506:   9% ▕█                 ▏ 414 MB/4.7 GB                  pulling manifest 
pulling 2bada8a74506:  11% ▕██                ▏ 525 MB/4.7 GB                  pulling manifest 
pulling 2bada8a74506:  14% ▕██                ▏ 634 MB/4.7 GB                  pulling manifest 
pulling 2bada8a74506:  15% ▕██                ▏ 684 MB/4.7 GB    

In [3]:
import ollama
r = ollama.chat(model=LLM, messages=[{"role": "user", "content": "Xin chào, bạn là ai?"}])
print(r["message"]["content"])

Xin chào! Tôi là Qwen, một mô hình ngôn ngữ lớn được tạo ra bởi Alibaba Cloud. Tôi có thể giúp bạn với nhiều loại công việc liên quan đến ngôn ngữ tự nhiên, như viết câu chuyện, email, kịch bản, hoặc thậm chí là các ý tưởng sáng tạo khác.


In [4]:
import glob
from datasets import load_dataset
from pypdf import PdfReader

docs = []   # mỗi phần tử: {"head", "body", "src", "lang", "link"}

def pick(cols, cands):
    for c in cands:
        if c in cols:
            return c
    return None

def add_qa(name, config=None, splits=("train",), lang="vi", limit=None,
           q_cands=("question", "Question", "input"),
           a_cands=("answer", "Answer", "output", "long_answer")):
    """Nạp 1 dataset hỏi đáp từ Hugging Face; lỗi thì bỏ qua, không làm hỏng cả pipeline."""
    try:
        dd = load_dataset(name, config) if config else load_dataset(name)
    except Exception as e:
        print(f"❌ Bỏ qua {name}: {str(e)[:150]}")
        return
    n = 0
    for sp in splits:
        if sp not in dd:
            continue
        ds = dd[sp]
        qc, ac = pick(ds.column_names, q_cands), pick(ds.column_names, a_cands)
        if not qc or not ac:
            print(f"❌ {name}: không thấy cột hỏi/đáp. Các cột: {ds.column_names}")
            return
        for row in ds:
            if limit and n >= limit:
                break
            q, a = str(row[qc] or "").strip(), str(row[ac] or "").strip()
            if len(q) < 5 or len(a) < 5:
                continue
            docs.append({"head": f"Q: {q}\nA:", "body": a,
                         "src": name.split("/")[-1], "lang": lang,
                         "link": str(row.get("link", "") or "")})
            n += 1
    print(f"✅ {name}: {n} cặp hỏi đáp")

def add_files(folder="/kaggle/input"):
    """Nạp PDF/TXT/MD do bạn tải lên Kaggle."""
    n = 0
    for f in glob.glob(f"{folder}/**/*.pdf", recursive=True):
        try:
            for i, p in enumerate(PdfReader(f).pages):
                t = (p.extract_text() or "").strip()
                if t:
                    docs.append({"head": "", "body": t, "src": f"{f.split('/')[-1]} - trang {i+1}",
                                 "lang": "vi", "link": ""}); n += 1
        except Exception as e:
            print("Lỗi đọc", f, e)
    for ext in ("txt", "md"):
        for f in glob.glob(f"{folder}/**/*.{ext}", recursive=True):
            t = open(f, encoding="utf-8", errors="ignore").read().strip()
            if t:
                docs.append({"head": "", "body": t, "src": f.split("/")[-1],
                             "lang": "vi", "link": ""}); n += 1
    print(f"✅ File của khoa: {n} đoạn/trang")

# --- Tiếng Việt ---
add_qa("tarudesu/ViHealthQA", splits=("train", "validation", "test"), lang="vi")
add_qa("hungnm/vietnamese-medical-qa", lang="vi")

# --- Tiếng Anh (giữ nguyên bản, LLM sẽ trả lời bằng tiếng Việt) ---
add_qa("keivalya/MedQuad-MedicalQnADataset", lang="en", limit=6000)
add_qa("medalpaca/medical_meadow_wikidoc", lang="en", limit=4000)
add_qa("qiaojin/PubMedQA", config="pqa_labeled", lang="en")

# --- File riêng của khoa ---
add_files()

print("\nTổng:", len(docs), "tài liệu")

README.md:   0%|          | 0.00/2.66k [00:00<?, ?B/s]

train.csv:   0%|          | 0.00/6.29M [00:00<?, ?B/s]

val.csv:   0%|          | 0.00/886k [00:00<?, ?B/s]

test.csv:   0%|          | 0.00/1.86M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7009 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/993 [00:00<?, ? examples/s]

... (18 outputs left out)

pqa_labeled/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.08MB            

pqa_labeled/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

✅ qiaojin/PubMedQA: 1000 cặp hỏi đáp
✅ File của khoa: 0 đoạn/trang

Tổng: 30350 tài liệu


In [5]:
import os, json, faiss, numpy as np
from sentence_transformers import SentenceTransformer

emb = SentenceTransformer("BAAI/bge-m3", device="cuda")
emb.max_seq_length = 512
emb.half()

def split_text(text, size=1000, overlap=150):
    return [text[i:i+size] for i in range(0, max(len(text), 1), size - overlap)]

chunks = []
for d in docs:
    pieces = [d["body"]] if len(d["head"]) + len(d["body"]) <= 1200 else split_text(d["body"])
    for p in pieces:
        chunks.append({"text": (d["head"] + "\n" + p).strip(),
                       "src": d["src"], "lang": d["lang"], "link": d["link"]})
print(len(chunks), "đoạn")

CACHE_X, CACHE_C = "/kaggle/working/X.npy", "/kaggle/working/chunks.json"
if os.path.exists(CACHE_X) and os.path.exists(CACHE_C):
    X = np.load(CACHE_X)
    chunks = json.load(open(CACHE_C, encoding="utf-8"))
    print("Đã nạp cache")
else:
    X = emb.encode([c["text"] for c in chunks], batch_size=64,
                   normalize_embeddings=True, show_progress_bar=True).astype("float32")
    np.save(CACHE_X, X)
    json.dump(chunks, open(CACHE_C, "w", encoding="utf-8"), ensure_ascii=False)

index = faiss.IndexFlatIP(X.shape[1])
index.add(X)
print("Index:", index.ntotal)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 2.27GB            

... (10 outputs left out)

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

43604 đoạn


Batches:   0%|          | 0/682 [00:00<?, ?it/s]

Index: 43604


In [6]:
import ollama

SYSTEM = """Bạn là trợ lý hỏi đáp y khoa của khoa. Quy tắc:
1. Chỉ trả lời dựa trên NGỮ CẢNH được cung cấp. Nếu ngữ cảnh không đủ thông tin, nói: "Mình chưa tìm thấy thông tin này trong tài liệu."
2. Ngữ cảnh có thể bằng tiếng Anh: hãy đọc hiểu và LUÔN trả lời bằng tiếng Việt tự nhiên. Thuật ngữ y khoa quan trọng ghi kèm tiếng Anh trong ngoặc.
3. Không tự bịa liều thuốc, chẩn đoán hay số liệu không có trong ngữ cảnh.
4. Trả lời ngắn gọn, rõ ràng. Kết thúc bằng một câu nhắc: thông tin chỉ mang tính tham khảo, cần hỏi bác sĩ khi có triệu chứng."""

def retrieve(question, k=5, min_score=0.3):
    v = emb.encode([question], normalize_embeddings=True).astype("float32")
    scores, idx = index.search(v, k)
    return [(chunks[i], float(s)) for s, i in zip(scores[0], idx[0]) if i != -1 and s >= min_score]

def answer_stream(question, k=5):
    hits = retrieve(question, k)
    if not hits:
        yield "Mình chưa tìm thấy thông tin này trong tài liệu."
        return

    context = "\n\n".join(f"[{i+1}] (nguồn: {h['src']})\n{h['text']}"
                          for i, (h, _) in enumerate(hits))
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": f"NGỮ CẢNH:\n{context}\n\nCÂU HỎI: {question}"},
    ]

    out = ""
    for part in ollama.chat(model=LLM, messages=messages, stream=True,
                            options={"temperature": 0.2, "num_ctx": 6144}):
        out += part["message"]["content"]
        yield out

    seen, srcs = set(), []
    for h, _ in hits:
        label = h["src"] + (f" ({h['link']})" if h["link"] else "")
        if label not in seen:
            seen.add(label); srcs.append(label)
    yield out + "\n\n**Nguồn:** " + "; ".join(srcs)

In [7]:
for text in answer_stream("Đang chích ngừa viêm gan B có tiêm vaccine Covid-19 được không?"):
    pass
print(text)

for text in answer_stream("Các biến chứng thường gặp của đái tháo đường là gì?"):  # kiểm tra truy vấn sang tài liệu tiếng Anh
    pass
print(text)

Nếu bạn đang tiêm ngừa vaccine phòng bệnh viêm gan B, bạn vẫn có thể tiêm phòng vaccine phòng Covid-19, tuy nhiên vaccine Covid-19 phải được tiêm cách trước và sau mũi vaccine viêm gan B tối thiểu là 14 ngày.

Thông tin chỉ mang tính tham khảo, cần hỏi bác sĩ khi có triệu chứng.

**Nguồn:** ViHealthQA (https://vnexpress.net/tu-van-tiem-vaccine-covid-19-an-toan-hieu-qua-p85); ViHealthQA (https://vnexpress.net/tu-van-tiem-vaccine-covid-19-an-toan-hieu-qua-p121); ViHealthQA (https://vnexpress.net/tu-van-tiem-vaccine-covid-19-an-toan-hieu-qua-p22); ViHealthQA (https://vnexpress.net/tu-van-tiem-vaccine-covid-19-an-toan-hieu-qua-p57); ViHealthQA (https://vnexpress.net/tu-van-tiem-vaccine-covid-19-an-toan-hieu-qua-p43)
Chào bạn!

Theo ngữ cảnh, mẹ bạn 55 tuổi mắc tiểu đường type 2 đã 10 năm và có triệu chứng khó ngủ, cào ruột vào buổi tối. Đây có thể là do các biến chứng của tiểu đường, cụ thể là biến chứng thần kinh ngoại vi. Tuy nhiên, không thể xác định chính xác nguyên nhân mà không có th

In [ ]:
import gradio as gr

def chat_fn(message, history):
    yield from answer_stream(message)

gr.ChatInterface(
    chat_fn,
    title="Chatbot hỏi đáp Y khoa - Khoa",
    description="Thông tin chỉ mang tính tham khảo, không thay thế tư vấn của bác sĩ.",
    examples=["Tiêm vaccine viêm gan B xong có cần kiêng gì không?",
              "Triệu chứng của bệnh đái tháo đường là gì?"],
).launch(share=True, debug=True)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://c2fc4ede45d0757cea.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
